In [0]:
%sql
use catalog deltalake_catalog;

In [0]:
landing_zone = "/Volumes/deltalake_catalog/default/raw"
orders_data = landing_zone + "/ordershistory"
checkpoint_path = landing_zone + "/orders_checkpoint"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType

orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("order_date", TimestampType(), True),   
    StructField("customer_id", IntegerType(), True),
    StructField("order_status", StringType(), True),
])

In [0]:
ordersdf = (spark.readStream 
.format("cloudFiles") 
.option("cloudFiles.format", "csv") 
.option("header", "true")
.option("rescuedDataColumn", "_rescued_data")
.schema(orders_schema)
.option("cloudFiles.schemaLocation", checkpoint_path) 
.load(orders_data))



In [0]:
%sql
drop table if exists deltalake_catalog.default.ordersdelta;

In [0]:
ordersdf.writeStream \
.format("delta") \
.option("checkpointLocation", checkpoint_path) \
.option("mergeSchema", True) \
.outputMode("append") \
.trigger(availableNow=True) \
.toTable("deltalake_catalog.default.ordersdelta")



In [0]:
%sql
select * from deltalake_catalog.default.ordersdelta;

order_id,order_date,customer_id,order_status,_rescued_data
100000,2013-07-25T00:00:00.000Z,11599,CLOSED,"{""order_amount"":""10"",""_file_path"":""/Volumes/deltalake_catalog/default/raw/ordershistory/orders3.csv""}"
200000,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,"{""order_amount"":""20"",""_file_path"":""/Volumes/deltalake_catalog/default/raw/ordershistory/orders3.csv""}"
300000,2013-07-25T00:00:00.000Z,12111,COMPLETE,"{""order_amount"":""30"",""_file_path"":""/Volumes/deltalake_catalog/default/raw/ordershistory/orders3.csv""}"
400000,2013-07-25T00:00:00.000Z,8827,CLOSED,"{""order_amount"":""40"",""_file_path"":""/Volumes/deltalake_catalog/default/raw/ordershistory/orders3.csv""}"
5555555,2013-07-25T00:00:00.000Z,11599,CLOSED,null
6666666,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,null
7777777,2013-07-25T00:00:00.000Z,null,COMPLETE,"{""customer_id"":""COMPLETE"",""_file_path"":""/Volumes/deltalake_catalog/default/raw/ordershistory/orders4.csv""}"
8888888,2013-07-25T00:00:00.000Z,8827,CLOSED,null
1111111,2013-07-25T00:00:00.000Z,11599,CLOSED,null
1111111,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,null
